# Langfuse-style tracing — spans, scores, in-memory recorder

Once an agent crosses ~3 LLM/tool calls per request, print-debugging stops scaling. You need:

* **Spans** — named, timed segments with `input`, `output`, parent chain.
* **Scores** — numeric / boolean / categorical labels attached to spans.
* **A backend** — Langfuse (self-hosted), OpenTelemetry, Phoenix, Arize.

This notebook uses an in-memory recorder that matches `langfuse.Langfuse`'s shape so you can swap in the real client with one line.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
from recorder import InMemoryTracer
from shared.embedders import cosine_topk, hash_embed
from shared.loaders import load_corpus, load_golden_qa

DOCS = load_corpus()
QA = {q.id: q for q in load_golden_qa()}
doc_texts = [d.title + '. ' + d.abstract for d in DOCS]
doc_ids = [d.arxiv_id for d in DOCS]
doc_vecs = hash_embed(doc_texts, dims=256, seed=0)
tracer = InMemoryTracer()
print('tracer ready')

## Instrument a tiny RAG pipeline

In [ ]:
def rag(question_id):
    item = QA[question_id]
    with tracer.trace('rag_request', question_id=item.id) as t:
        with tracer.span('retrieve') as r:
            qv = hash_embed([item.question], dims=256, seed=0)[0]
            idx, _ = cosine_topk(qv, doc_vecs, k=3)
            retrieved = [doc_ids[i] for i in idx]
            r.update(input=item.question, output=retrieved)
            r.score('recall_at_k_hit', bool(set(retrieved) & set(item.source_ids)))
        with tracer.span('generate') as g:
            answer = f'[{retrieved[0]}] would-go-here'
            g.update(input=retrieved, output=answer)
        t.update(output=answer)
        t.score('trace_ok', True)

for qid in ('q01', 'q05', 'q23'):
    rag(qid)
print(f'recorded {len(tracer.traces)} traces, {len(tracer.spans)} spans')

## Inspect the recorded structure

In [ ]:
for trace_id, spans in tracer.traces.items():
    print(f'--- trace {trace_id[:8]} ---')
    by_id = {s.span_id: s for s in spans}
    for s in spans:
        depth = 0
        cur = s
        while cur.parent_id is not None and cur.parent_id in by_id:
            depth += 1
            cur = by_id[cur.parent_id]
        indent = '  ' * depth
        scores = ' '.join(f'{sc.name}={sc.value}' for sc in s.scores)
        print(f'{indent}{s.name} ({s.duration_ms:.2f} ms) {scores}')

## Swap in the real Langfuse client

Change one import:

```python
from langfuse import Langfuse
tracer = Langfuse(host='http://localhost:3000', public_key=..., secret_key=...)
```

All the `tracer.trace(...)` / `tracer.span(...)` / `r.score(...)` calls in the RAG function above are intentionally name-compatible.

We'll deploy real Langfuse via Docker Compose in `07-deployment-patterns/02-docker-compose-stack/` (Phase 8).